ЗА ВАЛИДНИ ДУМИ
1. Зареждане на стринга
2. Премахване на специални символи
3. Разделяне на думи
4. Проверка на всяка дума дали съществува в речника
5. Ако да добавяне на думата в правилния израз
6. Ако не подаване на невронната мрежа за разпознаване

ДАННИ ЗА НЕВРОНЕН МОДЕЛ
1. Кодиране на всяка дума в речника:
  /а. буква позиция - за позиция в голям текст
  /б. тегло на буква - пореден номер в азбуката / 10 . позиция
  тегла: а-1, б-2, в-3, г-4, ....
  позиция: 0,1,2,3, ... - превръщане в бинарен код с добавяне на единица в края за да се включат нулите в кода които се падат в края и се отрязват от десетичната стойност: 1110 -> 11101
  Пример: тест
2. Генериране на грешни думи и създаване на бачове за всяка
3. Създаване на тренировъчния и тестовия сет
4. Трениране на модела


НЕВРОНЕН МОДЕЛ
1. ЛСТМ  базов
2. Входен слой 3 параметъра:
  2.1. кодиране по максимална дължина - използване на кодировката на думата запълвайки остатъка с 0 до максималната зададена дължина на дума, пример: 
  дума = коклюш, 
  кодировка = [1.101, 1.511, 1.1101, 1.2111, 2.9100099999999998, 2.51011]
  допълване = [1.101, 1.511, 1.1101, 1.2111, 2.9100099999999998, 2.51011, 0, 0, 0, .... MAX_LEN]
  2.2. контролното число
  2.3. кодирока на наставките 

In [47]:
import math
MAX_WORD_LEN = 30
letters = ['а', 'б', 'в', 'г', 'д', 'е', 'ж', 'з', 'и', 'й', 
           'к', 'л', 'м', 'н', 'о', 'п', 'р', 'с', 'т', 'у', 
           'ф', 'х', 'ц', 'ч', 'ш', 'щ', 'ъ', 'ь', 'ю', 'я',]
alpha_weight = lambda word: [(letters.index(alpha)+1)/10 if alpha in letters else 0. for alpha in word ]
alpha_position = lambda index: [int(bit) for bit in bin(index)[2:]]
def alpha_code(code:list):
    point_10 = 0.
    exp = 2
    for bit in code:
        point_10 += bit/(10**exp)
        exp+=1
    point_10 += 1/(10**exp)
    return point_10
def word_code(word:str):
    alpha_nums = alpha_weight(word)
    for i in range(len(alpha_nums)):
        position = alpha_position(i)
        if alpha_nums[i] == 0.: 
            print('zero')
            continue
        alpha_nums[i] += alpha_code(position)
    return alpha_nums

In [48]:
word = 'коклюш' #
word = word.lower()
code = word_code(word)
code_crc = 1
for c in code:
    code_crc *= c
code_crc

16.33733165061446

In [49]:
code

[1.101, 1.511, 1.1101, 1.2111, 2.9100099999999998, 2.51011]

In [44]:
class Node:
    
    def __init__(self, word) -> None:
        self.word = word
        self.right = None
        self.left = None
        self.adjacent_dictionary = {}

    def insert(self, word):
        if word <= self.word:
            if self.left is None:
                self.left = Node(word)
            else:
                self.left.insert(word)
        else:
            if self.right is None:
                self.right = Node(word)
            else:
                self.right.insert(word)
    #create dictionary with nodes and their children
    def create_adjacent(self,node):
        if node is None: return
        self.adjacent_dictionary[node.word] = []
        
        if node.left is not None: self.adjacent_dictionary[node.word].append(node.left.word)
        if node.right is not None: self.adjacent_dictionary[node.word].append(node.right.word)
        
        self.create_adjacent(node.left)
        self.create_adjacent(node.right)

    def breadth_first_search(self):
        if not self.adjacent_dictionary: self.create_adjacent(self)
        queue = [self.word,]
        visited = []
        while queue:
            node = queue.pop(0)
            visited.append(node)
            [queue.append(word) for word in self.adjacent_dictionary[node]]
        return visited
    
    @staticmethod
    def in_order(node):
        if node is None: return
        if not isinstance(node,Node): return
        Node.in_order(node.left)
        print(node.word, end=' ')
        Node.in_order(node.right)

    @staticmethod
    def pre_order(node):
        if node is None: return
        if not isinstance(node,Node): return
        print(node.word, end=' ')
        Node.pre_order(node.left)
        Node.pre_order(node.right)

    @staticmethod
    def post_order(node):
        if node is None: return
        if not isinstance(node,Node): return
        print(node.word, end=' ')
        Node.post_order(node.right)
        Node.post_order(node.left)

In [45]:
words = ['боза', 'мляко', 'желязо', 'земя', 'огън', 'свобода', 'вярност', 'хляб', 'вода', 'небе']
word_weights = []
root = Node('аз')
for word in words:
    root.insert(word)
    code_list = word_code(word)
    word_weights.append(math.prod(code_list))


In [5]:
code_list

[1.4009999999999998, 0.611, 0.2101, 0.6111]

In [30]:
Node.pre_order(root)
print()
Node.in_order(root)
print()
Node.post_order(root)

аз боза мляко желязо вярност вода земя огън небе свобода хляб 
аз боза вода вярност желязо земя мляко небе огън свобода хляб 
аз боза мляко огън свобода хляб небе желязо земя вярност вода 

In [31]:
root.create_adjacent(root)

In [32]:
root.adjacent_dictionary

{'аз': ['боза'],
 'боза': ['мляко'],
 'мляко': ['желязо', 'огън'],
 'желязо': ['вярност', 'земя'],
 'вярност': ['вода'],
 'вода': [],
 'земя': [],
 'огън': ['небе', 'свобода'],
 'небе': [],
 'свобода': ['хляб'],
 'хляб': []}

In [46]:
root.breadth_first_search()

['аз',
 'боза',
 'мляко',
 'желязо',
 'огън',
 'вярност',
 'земя',
 'небе',
 'свобода',
 'вода',
 'хляб']

In [43]:
root.adjacent_dictionary

{'аз': ['боза'],
 'боза': ['мляко'],
 'мляко': ['желязо', 'огън'],
 'желязо': ['вярност', 'земя'],
 'вярност': ['вода'],
 'вода': [],
 'земя': [],
 'огън': ['небе', 'свобода'],
 'небе': [],
 'свобода': ['хляб'],
 'хляб': []}